# A股盘后复盘分析

**用途**: 每日收盘后深度分析市场表现

**数据源**: 新浪财经 API + 东方财富 API + Yahoo Finance

**运行方式**: 按顺序执行所有单元格（Run All）

In [ ]:
# ========================================
# Setup: 导入库和工具函数
# ========================================
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from datetime import datetime

# 中文显示设置
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)

UA = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'

print(f'分析时间: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

## 1. A股主要指数行情

从新浪财经获取7大指数实时数据

In [ ]:
# 新浪财经 API — 获取A股指数
INDEX_CODES = 's_sh000001,s_sh000300,s_sh000016,s_sz399001,s_sz399006,s_sh000905,s_sh000688'
INDEX_NAMES = ['上证指数', '沪深300', '上证50', '深证成指', '创业板指', '中证500', '科创50']

def fetch_sina_indices(codes):
    url = f'https://hq.sinajs.cn/?list={codes}'
    headers = {'Referer': 'https://finance.sina.com.cn', 'User-Agent': UA}
    r = requests.get(url, headers=headers, timeout=15)
    r.encoding = 'gbk'
    
    results = {}
    for line in r.text.split('\n'):
        if '="' in line:
            code = line.split('hq_str_')[1].split('="')[0]
            data = line.split('="')[1].rstrip('";').split(',')
            results[code] = data
    return results

data = fetch_sina_indices(INDEX_CODES)

indices = []
for code, name in zip(INDEX_CODES.split(','), INDEX_NAMES):
    f = data.get(code, [])
    if len(f) > 3:
        indices.append({
            'name': name,
            'price': float(f[1]),
            'changeAmt': float(f[2]),
            'changePct': float(f[3]),
            'volume_yi': float(f[5]) / 10000 if len(f) > 5 else 0,
        })

for i in indices:
    arrow = '↑' if i['changePct'] > 0 else '↓'
    print(f"{i['name']:6s}  {i['price']:10.2f}  {arrow}{abs(i['changePct']):.2f}%  ({i['changeAmt']:+.2f})")

print(f"\n共 {len(indices)} 个指数")

## 2. 指数涨跌幅可视化

水平条形图 — 红涨绿跌

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

names = [i['name'] for i in indices]
pcts = [i['changePct'] for i in indices]
colors = ['#C44E52' if p < 0 else '#55A868' for p in pcts]

bars = ax.barh(names, pcts, color=colors, edgecolor='white', height=0.6)
for bar, pct in zip(bars, pcts):
    x = bar.get_width()
    ax.text(x + 0.05 if x >= 0 else x - 0.05, bar.get_y() + bar.get_height()/2,
            f'{pct:+.2f}%', ha='left' if x >= 0 else 'right', va='center', fontsize=11)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'A股主要指数涨跌幅 ({datetime.now().strftime("%Y-%m-%d")})', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 3. 行业板块热度

从东方财富获取板块涨跌排行

In [ ]:
# 东方财富 API — 行业板块
EM_UT = 'b2884a393a59ad64002292a3e90d46a5'
EM_HEADERS = {
    'User-Agent': UA,
    'Referer': 'https://quote.eastmoney.com/',
}

def fetch_sectors():
    params = {
        'pn': '1', 'pz': '100', 'po': '1', 'np': '1',
        'fid': 'f3', 'fs': 'm:90+t:2',
        'fields': 'f3,f4,f12,f14,f104,f105',
        'ut': EM_UT,
    }
    # 先获取 cookie
    s = requests.Session()
    s.get('https://quote.eastmoney.com/', headers=EM_HEADERS, timeout=15)
    
    url = 'https://push2.eastmoney.com/api/qt/clist/get'
    r = s.get(url, params=params, headers=EM_HEADERS, timeout=15)
    data = r.json()
    
    items = data.get('data', {}).get('diff', [])
    # 只保留一级分类（不含II/III后缀）
    items = [s for s in items if s.get('f14') and 'Ⅱ' not in s['f14'] and 'Ⅲ' not in s['f14']]
    return [{
        'name': s['f14'],
        'changePct': float(s['f3']) / 100,
        'up': int(s.get('f104', 0) or 0),
        'down': int(s.get('f105', 0) or 0),
    } for s in items]

sectors = fetch_sectors()
sectors.sort(key=lambda s: s['changePct'], reverse=True)

print('Top 5 领涨板块:')
for s in sectors[:5]:
    print(f"  {s['name']:10s}  +{s['changePct']:.2f}%")
print('\nBottom 5 领跌板块:')
for s in sectors[-5:]:
    print(f"  {s['name']:10s}  {s['changePct']:.2f}%")

## 4. 板块热力图

展示 Top 10 涨跌幅板块

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

top_n = 10
display = sectors[-top_n:] + sectors[:top_n][::-1]
names = [s['name'] for s in display]
pcts = [s['changePct'] for s in display]
colors = ['#C44E52' if p < 0 else '#55A868' for p in pcts]

bars = ax.barh(names, pcts, color=colors, edgecolor='white', height=0.6)
for bar, pct in zip(bars, pcts):
    x = bar.get_width()
    ax.text(x + 0.03 if x >= 0 else x - 0.03, bar.get_y() + bar.get_height()/2,
            f'{pct:+.2f}%', ha='left' if x >= 0 else 'right', va='center', fontsize=9)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'行业板块涨跌 Top{top_n} ({datetime.now().strftime("%Y-%m-%d")})', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 5. 海外市场 + 估值

通过 Yahoo Finance 获取美股 ETF 估值数据

In [ ]:
try:
    import yfinance as yf
    
    etfs = [
        ('SPY', '标普500'),
        ('QQQ', '纳斯达克100'),
        ('DIA', '道琼斯'),
        ('IWM', '罗素2000'),
    ]
    
    print(f"{'名称':12s} {'价格':>10s} {'PE':>8s} {'52周高':>10s} {'52周低':>10s}")
    print('-' * 55)
    
    for ticker, name in etfs:
        t = yf.Ticker(ticker)
        info = t.info
        price = info.get('regularMarketPrice', info.get('previousClose', '—'))
        pe = info.get('trailingPE', '—')
        hi = info.get('fiftyTwoWeekHigh', '—')
        lo = info.get('fiftyTwoWeekLow', '—')
        
        pe_str = f'{pe:.1f}' if isinstance(pe, (int, float)) else '—'
        hi_str = f'{hi:.2f}' if isinstance(hi, (int, float)) else '—'
        lo_str = f'{lo:.2f}' if isinstance(lo, (int, float)) else '—'
        price_str = f'{price:.2f}' if isinstance(price, (int, float)) else '—'
        
        print(f"{name:12s} {price_str:>10s} {pe_str:>8s} {hi_str:>10s} {lo_str:>10s}")
    
    # VIX
    vix = yf.Ticker('^VIX')
    vix_price = vix.info.get('regularMarketPrice', '—')
    print(f"\nVIX 恐慌指数: {vix_price}")
    
except ImportError:
    print('yfinance 未安装。运行: pip install yfinance')
except Exception as e:
    print(f'获取海外数据失败: {e}')

## 6. 总结

- 今日市场核心特征：
- 领涨 / 领跌板块：
- 海外市场影响：
- 明日关注点：

---
*数据来源: 新浪财经 + 东方财富 + Yahoo Finance*

## 扩展：从晨报 JSON 文件快速分析

如果已运行过 `morning-report.js`，可直接读取 `market-data.json` 快速分析

In [ ]:
import json, os

json_path = '../market-data.json'
if os.path.exists(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print('已加载 market-data.json')
    print(f"  A股指数: {len(data.get('indices', []))} 个")
    print(f"  海外指数: {len(data.get('overseas', []))} 个")
    print(f"  外汇对: {len(data.get('forexList', []))} 个")
    print(f"  商品: {len(data.get('commodities', []))} 个")
    print(f"  新闻: {len(data.get('articles', []))} 条")
    
    # AI 总结
    summary = data.get('aiSummary', '')
    if summary:
        print(f"\nAI 市场总结:\n{summary}")
else:
    print('未找到 market-data.json，请先运行 node morning-report.js --fetch')